# PySpark DataFrame Data Engineering Project — Real Estate Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `properties.csv`, `property_sales.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("properties.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("property_sales.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select property_id and city from the first DataFrame.

In [ ]:
df1.select('property_id','city').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where sale_price is greater than its average.

In [ ]:
avg_value = df2.select(avg('sale_price').alias('avg')).first()['avg']
df2.filter(col('sale_price') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on sale_price using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('sale_price') >= avg('sale_price').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in city and property_type with 'Unknown'.

In [ ]:
df1.fillna({'city': 'Unknown'}).show(5)
df2.fillna({'property_type': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using property_id and property_id.

In [ ]:
joined = df1.join(df2, df1['property_id'] == df2['property_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['property_id'] == df2['property_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average sale_price by property_type.

In [ ]:
df2.groupBy('property_type').agg(count('*').alias('records'), avg('sale_price').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total sale_price by property_type.

In [ ]:
df2.groupBy('property_type').agg(sum('sale_price').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum sale_price in each property_type.

In [ ]:
df2.groupBy('property_type').agg(max('sale_price').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum sale_price in each property_type.

In [ ]:
df2.groupBy('property_type').agg(min('sale_price').alias('min_value')).show()

## Q14. Having Condition

Show property_type groups whose total sale_price is greater than the overall average sale_price.

In [ ]:
overall = df2.select(avg('sale_price').alias('v')).first()['v']
df2.groupBy('property_type').agg(sum('sale_price').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by sale_price.

In [ ]:
df2.orderBy(desc('sale_price')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by sale_price.

In [ ]:
df2.orderBy(asc('sale_price')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by sale_price within each property_type.

In [ ]:
w = Window.partitionBy('property_type').orderBy(desc('sale_price'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of sale_price within each property_type.

In [ ]:
w = Window.partitionBy('property_type').orderBy('sale_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('sale_price').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's sale_price with the previous record in its property_type.

In [ ]:
w = Window.partitionBy('property_type').orderBy('sale_date')
df2.withColumn('previous_value', lag('sale_price').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each property_type's contribution to total sale_price.

In [ ]:
tot = df2.select(sum('sale_price').alias('t')).first()['t']
df2.groupBy('property_type').agg(sum('sale_price').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from sale_date.

In [ ]:
df2.withColumn('year', year('sale_date')).withColumn('month', month('sale_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average sale_price by month.

In [ ]:
df2.withColumn('month', date_format('sale_date','yyyy-MM')).groupBy('month').agg(sum('sale_price').alias('total'), avg('sale_price').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with property_type as rows and year of sale_date as columns.

In [ ]:
df2.withColumn('year', year('sale_date')).groupBy('property_type').pivot('year').agg(sum('sale_price')).show()

## Q24. Conditional KPI

Count records where sale_price is above the category average.

In [ ]:
w=Window.partitionBy('property_type')
df2.withColumn('cat_avg',avg('sale_price').over(w)).filter(col('sale_price')>col('cat_avg')).groupBy('property_type').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total sale_price by city.

In [ ]:
df1.join(df2, df1['property_id']==df2['property_id']).groupBy(df1['city']).agg(sum(df2['sale_price']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank city by total sale_price.

In [ ]:
r=df1.join(df2,df1['property_id']==df2['property_id']).groupBy(df1['city']).agg(sum(df2['sale_price']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for sale_price.

In [ ]:
df2.withColumn('segment',when(col('sale_price')<1000,'Low').when(col('sale_price')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative sale_price values.

In [ ]:
df2.filter(col('sale_price')<0).count()

## Q29. Final Business Report

Create a final report by property_type with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('property_type').agg(count('*').alias('records'),sum('sale_price').alias('total'),avg('sale_price').alias('average'),min('sale_price').alias('minimum'),max('sale_price').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project